# AMR Vision — YOLOv8n Custom Training
**Al-Shifa Medical Warehouse · UTB Capstone 2026**

This notebook trains a custom YOLOv8n (nano) model on our merged dataset.

### 3 Classes
| ID | Name | Where it appears |
|---|---|---|
| 0 | `box` | Cardboard boxes on warehouse shelves |
| 1 | `qr` | QR codes on the lower part of the shelf (shelf identity) |
| 2 | `barcode` | Barcodes printed on boxes (item SKU identity) |

### Before running this notebook
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Make sure `amr_dataset.zip` (or the `merged` folder zipped) is in your Google Drive
3. Run cells **top to bottom**, one at a time

---
## Cell 1 — Check GPU
Always run this first. If it says 'No GPU', stop and change the runtime before continuing.
Training on CPU instead of GPU would take 20x longer and would likely time out.

In [ ]:
import torch

print('=' * 45)
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {vram:.1f} GB')
else:
    print('WARNING: No GPU detected!')
    print('Go to: Runtime > Change runtime type > T4 GPU')
print('=' * 45)

---
## Cell 2 — Install Ultralytics YOLOv8

`ultralytics` is the official Ultralytics library. It handles everything:
- YOLOv8 model architecture (~3M parameters for nano)
- Training loop, optimizer, learning rate scheduler
- Data augmentation pipeline
- Evaluation metrics (mAP, precision, recall)
- Auto-downloading COCO pretrained weights to start from

The `-q` flag suppresses noisy output. `ultralytics.checks()` confirms GPU is visible to the library.

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics

# This prints version info and confirms CUDA device is found
ultralytics.checks()

---
## Cell 3 — Mount Google Drive and Extract Dataset

**Update `DATASET_ZIP`** to match where you saved your zip in Drive.

Examples:
- If you put it directly in Drive root: `/content/drive/MyDrive/amr_dataset.zip`
- If you put it in a folder called AMR: `/content/drive/MyDrive/AMR/amr_dataset.zip`

The extraction is skipped if the folder already exists (safe to re-run).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
from pathlib import Path

# ── CHANGE THIS to your actual zip path in Drive ──────────────────
DATASET_ZIP = '/content/drive/MyDrive/amr_dataset.zip'
# ─────────────────────────────────────────────────────────────────

DATASET_DIR = '/content/amr_dataset'

if not os.path.exists(DATASET_DIR):
    print(f'Extracting {DATASET_ZIP} ...')
    with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
        z.extractall(DATASET_DIR)
    print('Extraction complete.')
else:
    print('Dataset folder already exists — skipping extraction.')

print('\nTop-level contents:')
for item in sorted(os.listdir(DATASET_DIR)):
    print(f'  {item}')

---
## Cell 4 — Verify Dataset and Fix data.yaml Paths

**Why do we need to fix the paths?**

The `data.yaml` we generated on Windows uses relative paths like `../train/images`.
These relative paths work when the YAML file and the data folders are in the same location,
but on Colab, YOLOv8 needs **absolute paths** to find the data reliably.

We write a new file `/content/amr_data.yaml` with absolute Colab paths.

In [ ]:
import yaml

dataset_path = Path(DATASET_DIR)
colab_yaml   = '/content/amr_data.yaml'

# Read original yaml
with open(dataset_path / 'data.yaml') as f:
    data = yaml.safe_load(f)

# Replace relative paths with absolute Colab paths
data['train'] = str(dataset_path / 'train' / 'images')
data['val']   = str(dataset_path / 'valid' / 'images')
data['test']  = str(dataset_path / 'test'  / 'images')

with open(colab_yaml, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print('Updated data.yaml written to:', colab_yaml)
print()
with open(colab_yaml) as f:
    print(f.read())

# Count images per split to verify nothing is missing
print('Image counts:')
total = 0
for split, key in [('train', 'train'), ('valid', 'val'), ('test', 'test')]:
    img_dir = dataset_path / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.iterdir()))
        total += count
        print(f'  {split:6}: {count} images')
    else:
        print(f'  {split:6}: NOT FOUND')
print(f'  TOTAL : {total} images')

---
## Cell 5 — Train YOLOv8n

### What is transfer learning?
YOLOv8n was originally trained on the **COCO dataset** (80 classes, 118,000 images).
It already learned to detect edges, shapes, textures, and object boundaries.
We **fine-tune** it: we replace the final detection head (80 classes → 3 classes)
and continue training. This takes **hours instead of weeks** compared to training from scratch.

### Hyperparameter Guide

| Parameter | Value | Reason |
|---|---|---|
| `epochs` | 100 | Maximum iterations. Early stopping will likely end before 100. |
| `patience` | 20 | Stop if mAP50 does not improve for 20 consecutive epochs. |
| `imgsz` | 640 | Standard YOLO input resolution. Matches what Roboflow exported. |
| `batch` | 16 | Safe for T4 GPU (15GB VRAM). Increase to 32 if no memory error. |
| `hsv_v` | 0.6 | Strong brightness variation. Handles dim warehouse lighting. |
| `degrees` | 5.0 | Small rotation. Camera is mounted at 10-15 degrees tilt. |
| `mosaic` | 1.0 | Mixes 4 images into 1. Greatly improves small object detection. |
| `flipud` | 0.0 | No upside-down flip. Shelves are always upright in the real world. |
| `fliplr` | 0.5 | Random horizontal flip. Robot can approach from either side. |

### Estimated time
~1–2 hours on Colab T4 GPU for our dataset size (~9,000 training images, 100 epochs).
Early stopping typically triggers around epoch 60–80.

### What to watch
- `mAP50` column: your main quality metric. Target > 0.85 for all 3 classes.
- `box_loss` and `cls_loss`: should decrease steadily each epoch.
- If loss goes down then suddenly jumps back up: learning rate may be too high (not common with auto).

In [ ]:
from ultralytics import YOLO

# Load YOLOv8n pretrained on COCO
# yolov8n.pt (~6MB) auto-downloads if not cached
model = YOLO('yolov8n.pt')

print('Model loaded. Starting training...')
print('Watch the mAP50 column. Target: > 0.85')
print('Early stopping patience: 20 epochs')
print()

results = model.train(
    data          = colab_yaml,   # our fixed data.yaml with absolute paths
    epochs        = 100,          # max epochs
    patience      = 20,           # early stopping: stop if no improvement for 20 epochs
    imgsz         = 640,          # input resolution
    batch         = 16,           # batch size (safe for T4 15GB)
    name          = 'amr_vision', # results saved to runs/detect/amr_vision/
    exist_ok      = True,         # overwrite if run exists (safe to re-run cell)
    device        = 0,            # GPU 0
    optimizer     = 'auto',       # auto-select best optimizer (usually SGD then Adam)
    lr0           = 0.01,         # initial learning rate
    lrf           = 0.01,         # final lr = lr0 * lrf
    warmup_epochs = 3,            # slowly ramp up lr for first 3 epochs
    mosaic        = 1.0,          # mosaic augmentation strength
    hsv_h         = 0.015,        # hue shift (subtle color variation)
    hsv_s         = 0.7,          # saturation variation
    hsv_v         = 0.6,          # brightness variation (important for warehouse)
    degrees       = 5.0,          # rotation (matches camera tilt)
    fliplr        = 0.5,          # horizontal flip 50% of time
    flipud        = 0.0,          # no upside-down flip
    save_period   = 10,           # save checkpoint every 10 epochs
    val           = True,         # validate after each epoch
    plots         = True,         # save training curve plots
    verbose       = True,
)

RUN_DIR = str(results.save_dir)
print(f'\nTraining complete!')
print(f'Results saved to: {RUN_DIR}')
print(f'Best model: {RUN_DIR}/weights/best.pt')

---
## Cell 6 — View Training Curves

These plots are automatically generated by Ultralytics during training.

- **results.png**: All metrics over epochs (box_loss, cls_loss, mAP50, mAP50-95)
- **confusion_matrix.png**: Shows how often each class is confused with another
- **PR_curve.png**: Precision-Recall tradeoff for each class
- **F1_curve.png**: F1 score vs confidence threshold — useful for picking the best confidence threshold

A good confusion matrix for us: diagonal should be dark (most predictions correct), off-diagonal light.

In [ ]:
from IPython.display import Image, display
import os

# RUN_DIR was set in the training cell above
# If you re-ran training separately, set it manually:
# RUN_DIR = '/content/runs/detect/amr_vision'

plot_files = ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']

for fname in plot_files:
    fpath = os.path.join(RUN_DIR, fname)
    if os.path.exists(fpath):
        print(f'\n--- {fname} ---')
        display(Image(filename=fpath, width=900))
    else:
        print(f'[not found] {fpath}')

---
## Cell 7 — Validate the Best Model

### What do the metrics mean?

| Metric | Meaning | Good value for our use |
|---|---|---|
| **mAP50** | Average precision at IoU threshold 0.50 | > 0.85 |
| **mAP50-95** | Stricter: averaged over IoU 0.50 to 0.95 | > 0.60 |
| **Precision** | Of all boxes detected, % that were correct | > 0.85 |
| **Recall** | Of all real objects, % that were found | > 0.85 |

### For our robot, recall matters more than precision
- Missing a QR code = robot proceeds to wrong shelf (dangerous)
- Missing a barcode = robot picks wrong item (dangerous)
- False detection = extra verification step (acceptable)

So we prefer **high recall** even if precision drops slightly.

In [ ]:
from ultralytics import YOLO

best_model_path = os.path.join(RUN_DIR, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

print('Running validation on validation split...')
val_results = best_model.val(
    data   = colab_yaml,
    imgsz  = 640,
    batch  = 16,
    device = 0,
)

print('\n' + '=' * 40)
print('  VALIDATION RESULTS')
print('=' * 40)
print(f'  mAP50    : {val_results.box.map50:.4f}')
print(f'  mAP50-95 : {val_results.box.map:.4f}')
print(f'  Precision: {val_results.box.mp:.4f}')
print(f'  Recall   : {val_results.box.mr:.4f}')
print()
print('  Per-class mAP50:')
for i, name in enumerate(['box', 'qr', 'barcode']):
    if i < len(val_results.box.maps):
        print(f'    {name:10}: {val_results.box.maps[i]:.4f}')
print('=' * 40)

---
## Cell 8 — Test Inference on Sample Images

This runs the trained model on 6 random images from the test set and displays the results.
You should see green bounding boxes labeled `box`, `qr`, or `barcode` on the images.

**What to check:**
- Are all 3 class types being detected correctly?
- Are confidence scores reasonable (> 0.6 is good)?
- Are there false positives (detections where there is no object)?

In [ ]:
import glob, random
from IPython.display import Image, display

test_images = glob.glob(f'{DATASET_DIR}/test/images/*')

if not test_images:
    print('No test images found — using train images for visualization')
    test_images = glob.glob(f'{DATASET_DIR}/train/images/*')

# Pick 6 random images — run multiple times to see different samples
sample = random.sample(test_images, min(6, len(test_images)))

pred_dir = '/content/test_predictions'

# Clear previous predictions
import shutil
if os.path.exists(pred_dir):
    shutil.rmtree(pred_dir)

inf_results = best_model.predict(
    source  = sample,
    imgsz   = 640,
    conf    = 0.25,    # show detections with > 25% confidence
    iou     = 0.45,    # NMS overlap threshold
    save    = True,
    project = pred_dir,
    name    = 'results',
    verbose = False,
)

pred_images = sorted(
    glob.glob(f'{pred_dir}/results/*.jpg') +
    glob.glob(f'{pred_dir}/results/*.png')
)

print(f'Showing {len(pred_images)} predictions:')
for img_path in pred_images:
    display(Image(filename=img_path, width=640))

---
## Cell 9 — Save Model to Google Drive

**This is the most important cell — run it before closing Colab.**

Colab deletes all files when the session ends. Your trained `best.pt` will be lost
unless you save it to Drive.

The `best.pt` file is ~6MB. After saving:
1. Copy it to `ros2_ws/src/amr_vision/models/best.pt` on your laptop
2. Test it locally on your laptop with the Astra Pro (Task 4)
3. Later copy to the Jetson and convert to TensorRT (Task 8)

In [ ]:
import shutil
from google.colab import files

best_pt         = os.path.join(RUN_DIR, 'weights', 'best.pt')
drive_save_path = '/content/drive/MyDrive/amr_best.pt'

# 1. Save to Google Drive (persistent between sessions)
shutil.copy2(best_pt, drive_save_path)
print(f'Saved to Drive: {drive_save_path}')
print(f'Model size    : {os.path.getsize(best_pt) / 1e6:.2f} MB')

# 2. Also save the full training run folder to Drive
run_zip = '/content/amr_vision_run.zip'
shutil.make_archive('/content/amr_vision_run', 'zip', '/content/runs/detect', 'amr_vision')
shutil.copy2(run_zip, '/content/drive/MyDrive/amr_vision_run.zip')
print(f'Full run saved to Drive: /content/drive/MyDrive/amr_vision_run.zip')

# 3. Download best.pt directly to your laptop
print('\nStarting direct download to your laptop...')
files.download(best_pt)
print('Done. Move the downloaded file to:')
print('  AMR-Warehouse-System/ros2_ws/src/amr_vision/models/best.pt')

---
## Summary

When this notebook completes successfully you will have:

| File | Location | Purpose |
|---|---|---|
| `best.pt` | Google Drive + downloaded | Your custom-trained YOLOv8n model |
| `amr_vision_run.zip` | Google Drive | Full training run (curves, weights, config) |

### Next Steps
1. Copy `best.pt` to `models/best.pt` in your repo
2. **Task 4**: Test on your laptop with the Astra Pro camera
3. **Task 5**: Plug `best.pt` into the modular pipeline modules

### If mAP50 is low (< 0.70)
- Check the confusion matrix: which class is failing?
- If `barcode` is low: expected, we only had 562 training images — consider finding more
- If `qr` is low: increase `epochs` to 150 and lower confidence threshold to 0.2
- If `box` is low: unusual — check that label files have correct class ID 0